In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

from sklearn.metrics import classification_report
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import confusion_matrix

In [9]:
df = pd.read_csv("https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv")
df.to_csv('./data/penguins_clean.csv', index=False)
#df = pd.read_csv('./data/penguins.csv')
print(df.shape)
df.head()

(344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [12]:
df.dropna()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007
...,...,...,...,...,...,...,...,...
339,Chinstrap,Dream,55.8,19.8,207.0,4000.0,male,2009
340,Chinstrap,Dream,43.5,18.1,202.0,3400.0,female,2009
341,Chinstrap,Dream,49.6,18.2,193.0,3775.0,male,2009
342,Chinstrap,Dream,50.8,19.0,210.0,4100.0,male,2009


In [13]:
# Settings
N_TARGET = 5000          # change this for a different size
JITTER_SCALE = 0.05      # 0.00 for pure bootstrap; increase slightly to reduce duplicates
RANDOM_SEED = 20260311   # fixed for reproducibility
 
np.random.seed(RANDOM_SEED)
 
# Load & clean
df = pd.read_csv('./data/penguins_clean.csv')
df.rename(columns=lambda c: c.strip(), inplace=True)
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].astype(str).str.strip()
 
cat_cols = ['species','island','sex']
num_cols = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=num_cols)  # should already be clean
 
# Preserve joint proportions across categorical combo
g = df.groupby(cat_cols, dropna=False).size().reset_index(name='count')
g['prop']  = g['count'] / g['count'].sum()
g['alloc'] = np.floor(g['prop'] * N_TARGET).astype(int)
remainder  = N_TARGET - g['alloc'].sum()
g['frac']  = (g['prop'] * N_TARGET) - g['alloc']
for i in g['frac'].sort_values(ascending=False).index[:remainder]:
    g.at[i, 'alloc'] += 1
 
# Stratified bootstrap sampling
parts = []
for _, row in g.iterrows():
    mask = (df['species']==row['species']) & (df['island']==row['island']) & (df['sex']==row['sex'])
    gdf = df.loc[mask]
    idx = np.random.choice(gdf.index, size=int(row['alloc']), replace=True)
    parts.append(gdf.loc[idx])
syn = pd.concat(parts, axis=0, ignore_index=True)
 
# Gentle jitter per species, then clip to per-species ranges
spec_stats = {}
for sp, sdf in df.groupby('species'):
    spec_stats[sp] = {
        'min': sdf[num_cols].min(),
        'max': sdf[num_cols].max(),
        'std': sdf[num_cols].std(ddof=0).replace(0, 1e-6)
    }
for sp, idx in syn.groupby('species').groups.items():
    idx = list(idx)
    for c in num_cols:
        std = spec_stats[sp]['std'][c]
        noise = np.random.normal(0.0, std*JITTER_SCALE, size=len(idx))
        syn.loc[idx, c] = np.clip(syn.loc[idx, c].values + noise,
                                  spec_stats[sp]['min'][c],
                                  spec_stats[sp]['max'][c])
 
# Match measurement resolution
syn['bill_length_mm']   = syn['bill_length_mm'].round(1)
syn['bill_depth_mm']    = syn['bill_depth_mm'].round(1)
syn['flipper_length_mm']= syn['flipper_length_mm'].round().astype(int)
syn['body_mass_g']      = (25 * (syn['body_mass_g'] / 25).round()).astype(int)
 
# Save
syn.to_csv('./data/penguins_synthetic_5000.csv', index=False)
print(f'penguins_synthetic_5000.csv saved to /data/ folder')

penguins_synthetic_5000.csv saved to /data/ folder
